# Chapter 15 — Troubleshooting & Errors
> *When Git yells at you — here's exactly how to yell back.*
> *These 8 errors have ruined more mornings than alarm clocks.* 

---

>  The relationship between developers and Git error messages follows a predictable arc:
>
> Day 1: "What does this even mean. This error makes no sense."
> Week 2: "Oh this error. Right. It's that."
> Month 3: You start explaining this error to someone else effortlessly.
>
> You're about to skip Day 1 to Week 2 for all 8 of these. You're welcome.

## Error 1: `fatal: not a git repository`
```
fatal: not a git repository (or any parent directories): .git
```

**Translation:** You ran a Git command from OUTSIDE a Git repo. Git looked for `.git/` folder, found nothing.

```bash
pwd              # Mac/Linux/Git Bash: where are you right now?
Get-Location     # PowerShell: where are you right now?

# Navigate to your project:
cd /path/to/your/project
# Then try your Git command again

# OR if you're in a new folder and want to start a repo here:
git init
```

>  This one happens to every developer at least once a week.
> You're in the terminal, you type git status, it says "not a git repository."
> You're in the home directory. Or Desktop. Or Documents. Or anywhere except your project.
> `pwd` first. Always `pwd` first.

## Error 2: `Authentication failed` on Push
```
remote: Support for password authentication was removed on August 13, 2021.
fatal: Authentication failed for 'https://github.com/...'
```

**Translation:** You used your actual GitHub password. That hasn't worked since 2021. Use a PAT.
*(Full setup in [02_setup/02_authentication.ipynb](../02_setup/02_authentication.ipynb))*

```bash
# Clear bad stored credentials:
git config --global --unset credential.helper
# Then push again — enter GitHub username + PAT as password (NOT your real password)

# Windows: Start Menu → Credential Manager → Windows Credentials
# Find: git:https://github.com → Delete it → push again → enter PAT
```

>  This "authentication removed" message has caused more "is GitHub broken?"
> Stack Overflow posts than any other Git error.
> GitHub sent emails. GitHub had banners. GitHub announced it everywhere.
> People still hit this daily in 2026 because nobody reads announcements.
> Now you know. Use a PAT.

## Error 3: Push Rejected — `Updates were rejected`
```
! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/...'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. Integrate the remote changes (e.g. 'git pull ...') before
```

**Translation:** Someone else pushed while you were working. Sync first.

```bash
git pull origin main       # Get their commits first
git push origin main       # Now you can push

# Cleaner approach (rebase instead of merge commit):
git pull --rebase origin main
git push origin main
```

>  This is the classic "race condition" between developers.
> Two people working simultaneously. Whoever pushes first "wins."
> The second person gets this error.
> Pull first. Push second. Every time. Without exception.

## Error 4: `CONFLICT` — Merge Conflict
```
CONFLICT (content): Merge conflict in app.py
Automatic merge failed; fix conflicts and then commit the result.
```

**Translation:** Two branches changed the same line differently. Git needs YOU to decide which to keep.

```bash
# Find all conflicted files:
git status
# "both modified: app.py" ← each of these needs manual resolution

# Open the file, find the markers, resolve them:
# <<<<<<< HEAD
# your version
# =======
# their version
# >>>>>>> feature-branch
# DELETE the markers. KEEP the correct version (or combine both).

# Stage the resolved file:
git add app.py

# Complete the merge:
git commit

# OR — give up entirely and go back to before the merge:
git merge --abort
```

>  Merge conflicts feel catastrophic when they're new.
> After the 5th one, they're Tuesday.
> VS Code's conflict editor (Accept Current / Accept Incoming / Accept Both buttons)
> makes this 10x faster than editing raw markers.
> Open the file in VS Code. Click the buttons. Done. Move on.

## Error 5: `HEAD detached` State
```
HEAD detached at a3f8c2d
```

**Translation:** You checked out a specific commit instead of a branch. You're floating in history.
Nothing goes wrong if you just READ. But any commits you make here are in limbo.

```bash
# Escape back to safety:
git checkout main     # or git switch main

# If you accidentally made commits here and want to SAVE them:
git checkout -b rescue-branch        # Create a branch from detached HEAD
# Now those commits are on a real branch. They won't disappear.
```

>  Detached HEAD is not as alarming as it sounds.
> You're just "at a commit" instead of "at a branch."
> The moment you switch back to a branch, you're reattached.
> The only danger: accidentally making commits here and then switching away.
> Those commits enter Git limbo. Recoverable via reflog, but stressful.
> If you see "HEAD detached" — finish what you're looking at, then go back to a branch.

## Error 6: `Permission denied (publickey)` — SSH
```
git@github.com: Permission denied (publickey).
fatal: Could not read from remote repository.
```

**Translation:** SSH is trying to authenticate, but GitHub doesn't recognize your key.

```bash
# Check if your key is loaded in SSH agent:
ssh-add -l
# If "The agent has no identities" — key not loaded:

# Start SSH agent:
eval "$(ssh-agent -s)"

# Load your key:
ssh-add ~/.ssh/id_ed25519

# Test connection:
ssh -T git@github.com
# Success: "Hi username! You've successfully authenticated"
# Still failing: make sure your PUBLIC key is on GitHub
# GitHub Settings → SSH and GPG keys → verify your key is listed there
```

>  "No identities" means your SSH agent forgot your key.
> On Mac/Linux this often happens after a restart.
> Fix: `eval "$(ssh-agent -s)" && ssh-add ~/.ssh/id_ed25519`
> Add this to your shell startup file to make it permanent.

## Error 7: `src refspec main does not match any`
```
error: src refspec main does not match any
error: failed to push some refs to 'https://github.com/...'
```

**Translation:** You tried to push `main` but `main` doesn't exist yet — you have zero commits.

```bash
# Solution: make a commit first!
git add .
git commit -m "Initial commit"
git push -u origin main    # Now main exists, now you can push
```

>  Branches don't exist until you commit to them.
> `main` is just a label pointing to the latest commit.
> No commits = no label = "main does not match any."
> Create content first. Then push.

## Error 8: Line Ending Warning (Windows)
```
warning: LF will be replaced by CRLF in file.txt.
The file will have its original line endings in your working directory
```

**Translation:** Windows uses CRLF line endings. Unix/Mac uses LF. Git is normalizing them.
This is a WARNING, not an error. Your code still works. But it can cause fake "every line changed" noise in diffs.

```bash
# Fix it once for all repos on this machine:
git config --global core.autocrlf true    # Windows: normalize on commit
git config --global core.autocrlf input   # Mac/Linux: keep LF, never add CRLF
```

>  This warning was the first thing to confuse every Windows developer who ever used Git.
> You push your first commit. Your diff shows 200 lines changed. You changed 2 lines.
> Windows helpfully added `` to every single line.
> `core.autocrlf true` fixes this silently and forever.
> Set it. Forget it. Live your life.

---

##  When Everything Breaks — Emergency Checklist

```bash
git status           # What state are my files in? Start here.
git log --oneline    # What commits exist? What's the recent history?
git remote -v        # Is the remote URL correct?
git branch -vv       # What branch am I on? Is it tracking the right remote?
git reflog           # What has HEAD done recently? Recovery tool.
```

>  Most Git problems are solved by running these 5 commands in order.
> They give you the full picture. From full picture, cause is usually obvious.
> When in doubt: git status → git log → git reflog → Stack Overflow → coffee.